In [243]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
from weekly_team_predictions import *
from utils import *
pd.set_option('display.max_rows', None)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [244]:
schedules = build_schedules()
schedules.to_csv('schedules.csv', index=False)

In [245]:
features = assemble_features(schedules)
print("Number of features:", len(features))
print(features)

Number of features: 94
['spread_line', 'week', 'year', 'day_of_year', 'gametime', 'away_rest', 'home_rest', 'div_game', 'home_moneyline', 'away_moneyline', 'roof_closed', 'roof_dome', 'roof_open', 'roof_outdoors', 'surface_', 'surface_a_turf', 'surface_astroplay', 'surface_astroturf', 'surface_dessograss', 'surface_fieldturf', 'surface_grass', 'surface_grass ', 'surface_matrixturf', 'surface_sportturf', 'home_team_ARI', 'home_team_ATL', 'home_team_BAL', 'home_team_BUF', 'home_team_CAR', 'home_team_CHI', 'home_team_CIN', 'home_team_CLE', 'home_team_DAL', 'home_team_DEN', 'home_team_DET', 'home_team_GB', 'home_team_HOU', 'home_team_IND', 'home_team_JAX', 'home_team_KC', 'home_team_LA', 'home_team_LAC', 'home_team_LV', 'home_team_MIA', 'home_team_MIN', 'home_team_NE', 'home_team_NO', 'home_team_NYG', 'home_team_NYJ', 'home_team_OAK', 'home_team_PHI', 'home_team_PIT', 'home_team_SD', 'home_team_SEA', 'home_team_SF', 'home_team_STL', 'home_team_TB', 'home_team_TEN', 'home_team_WAS', 'away_t

In [246]:
#Train model 
def train_model(schedules: pd.DataFrame, features: list, seasons: list) -> tuple[LogisticRegression, StandardScaler, np.ndarray, pd.Series]:
    training_data = schedules.dropna(subset=features)
    training_data = training_data[training_data['year'].isin(seasons)]

    #Split into train and test sets
    X = training_data[features]
    y = training_data['result'] > 0 #Convert result from point dif to binary outcome 
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    #Scale features (put everything on scale 0-1)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #Train Model
    model = LogisticRegression(max_iter=10000)
    model.fit(X_train_scaled, y_train)  
    return model, scaler, X_test_scaled, y_test

# Test model accuracy results
def accuracy_report(y_test, y_pred):
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Model Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["Loss", "Win"]))

In [247]:
#build schedules
# assemble features
# train model
# get predictions
# build week1 and transform it
# convert week1 to account for weighted points based on outcome
SEASONS = list(range(2000, 2024))
model, scaler, X_test_scaled, y_test = train_model(schedules, features, seasons=SEASONS)
#Do prediction on test set
y_pred = model.predict(X_test_scaled)

In [248]:

WEEK = 2
YEAR = 2026

week_df, week_df_scaled = generate_inference_dataframe(WEEK, YEAR, schedules, scaler, features)
week_probs = generate_predictions(model, df_scaled=week_df_scaled)
print_predictions(week_df, week_probs)

Game: DET @ BUF, Predicted Win Probability for Home Team: 0.6437
Game: CAR @ ATL, Predicted Win Probability for Home Team: 0.4944
Game: NO @ BAL, Predicted Win Probability for Home Team: 0.7911
Game: MIN @ CHI, Predicted Win Probability for Home Team: 0.6426
Game: CIN @ HOU, Predicted Win Probability for Home Team: 0.7408
Game: PIT @ NE, Predicted Win Probability for Home Team: 0.7097
Game: GB @ NYJ, Predicted Win Probability for Home Team: 0.4507
Game: CLE @ TB, Predicted Win Probability for Home Team: 0.7996
Game: PHI @ TEN, Predicted Win Probability for Home Team: 0.2828
Game: JAX @ DEN, Predicted Win Probability for Home Team: 0.7206
Game: LV @ LAC, Predicted Win Probability for Home Team: 0.6180
Game: SEA @ ARI, Predicted Win Probability for Home Team: 0.4806
Game: WAS @ DAL, Predicted Win Probability for Home Team: 0.7767
Game: MIA @ SF, Predicted Win Probability for Home Team: 0.9177
Game: IND @ KC, Predicted Win Probability for Home Team: 0.6442
Game: NYG @ LA, Predicted Win Pr

In [249]:
results = generate_picks(week_df, week_probs)
print(results)

   favorite underdog  spread  favorite_win_prob  underdog_win_prob  \
0       BUF      DET     5.5           0.643694           0.356306   
1       CAR      ATL     2.5           0.505598           0.494402   
2       BAL       NO     8.5           0.791054           0.208946   
3       CHI      MIN     4.5           0.642571           0.357429   
4       HOU      CIN     3.0           0.740784           0.259216   
5        NE      PIT     4.5           0.709681           0.290319   
6        GB      NYJ     3.5           0.549338           0.450662   
7        TB      CLE     8.5           0.799591           0.200409   
8       PHI      TEN     7.0           0.717191           0.282809   
9       DEN      JAX     2.5           0.720566           0.279434   
10      LAC       LV     7.0           0.618013           0.381987   
11      SEA      ARI     3.5           0.519388           0.480612   
12      DAL      WAS     4.5           0.776706           0.223294   
13       SF      MIA

In [251]:
#Previous Year Comparison: get last years and figure out what score it would have made
YEAR = 2025
WEEK = None
previous_year, previous_year_scaled = generate_inference_dataframe(WEEK, YEAR, schedules, scaler, features)
predictions = generate_predictions(model, previous_year_scaled)
previous_year_results = generate_picks(previous_year, predictions)

previous_year_results['actual_winner'] = np.where(
    previous_year['result'] > 0,
    previous_year['home_team'],
    previous_year['away_team']
)

previous_year_points = np.where(
    previous_year_results['pick'] == previous_year_results['actual_winner'],
    np.where(
        previous_year_results['pick'] == previous_year_results['favorite'],
        1,
        np.where(
            previous_year_results['is_3pt_underdog'],
            3,
            2
        )
    ),
    0
)

print(f"Total Points for {YEAR}: {np.sum(previous_year_points)}")

#points if you picked perfectly (winners only)
perfect_points = np.where(
    previous_year_results['actual_winner'] == previous_year_results['favorite'],
    1,
    np.where(
        previous_year_results['is_3pt_underdog'],
        3,
        2
    )
)
print(f"Total Points if Picked Perfectly in {YEAR}: {np.sum(perfect_points)}")

previous_year_results.to_csv('2025.csv', index=False)

Total Points for 2025: 206
Total Points if Picked Perfectly in 2025: 383
